# YOLOv8n Object Detection Demo

All three protocols: REST JSON, gRPC, REST Binary

**Important:** Run cells in order. REST JSON must run first to avoid tritonclient state issues.

In [ ]:
import os
import time
import numpy as np
import cv2
import tritonclient.grpc as grpcclient
import tritonclient.http as httpclient

# =============================================================================
# Configuration - Update NAMESPACE for your environment
# =============================================================================
NAMESPACE = "domino-inference-dev"
#NAMESPACE = "domino-inference-test"
#NAMESPACE = "domino-inference-prod"

os.environ["TRITON_GRPC_URL"] = f"triton-inference-server-proxy.{NAMESPACE}.svc.cluster.local:50051"
os.environ["TRITON_REST_URL"] = f"http://triton-inference-server-proxy.{NAMESPACE}.svc.cluster.local:8080"

GRPC_URL = os.environ.get("TRITON_GRPC_URL", "localhost:50051")
REST_URL = os.environ.get("TRITON_REST_URL", "http://localhost:8080")

API_KEY = os.environ.get("DOMINO_USER_API_KEY", "")
grpc_headers = {"x-domino-api-key": API_KEY} if API_KEY else None
http_headers = {"X-Domino-Api-Key": API_KEY} if API_KEY else None

print(f"Namespace: {NAMESPACE}")
print(f"gRPC URL: {GRPC_URL}")
print(f"REST URL: {REST_URL}")
print(f"Auth: {'API Key configured' if API_KEY else 'Disabled'}")

In [ ]:
# =============================================================================
# Load and preprocess image
# =============================================================================
video_path = "../samples/video.avi"
cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

h, w = frame.shape[:2]
scale = min(640 / h, 640 / w)
new_w, new_h = int(w * scale), int(h * scale)
resized = cv2.resize(frame, (new_w, new_h))
img = np.full((640, 640, 3), 114, dtype=np.uint8)
pad_h, pad_w = (640 - new_h) // 2, (640 - new_w) // 2
img[pad_h:pad_h+new_h, pad_w:pad_w+new_w] = resized
base_tensor = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
base_tensor = np.transpose(base_tensor, (2, 0, 1))

print(f"Input shape: [1, {base_tensor.shape[0]}, {base_tensor.shape[1]}, {base_tensor.shape[2]}]")

url = REST_URL.replace("http://", "").replace("https://", "")

In [ ]:
# =============================================================================
# 1. REST (JSON) - MUST RUN FIRST
# =============================================================================
tensor_json = base_tensor[np.newaxis, ...].astype(np.float32).copy()
client = httpclient.InferenceServerClient(url=url)
inp = httpclient.InferInput("images", list(tensor_json.shape), "FP32")
inp.set_data_from_numpy(tensor_json, binary_data=False)
out = httpclient.InferRequestedOutput("output0", binary_data=False)

start = time.time()
resp = client.infer("yolov8n", [inp], outputs=[out], headers=http_headers)
t_json = (time.time() - start) * 1000
output_json = resp.as_numpy("output0")
print(f"✓ REST (JSON):   {t_json:8.2f} ms | Output: {output_json.shape}")

In [ ]:
# =============================================================================
# 2. gRPC Protocol
# =============================================================================
tensor_grpc = base_tensor[np.newaxis, ...].astype(np.float32).copy()
client = grpcclient.InferenceServerClient(url=GRPC_URL)
inp = grpcclient.InferInput("images", list(tensor_grpc.shape), "FP32")
inp.set_data_from_numpy(tensor_grpc)
out = grpcclient.InferRequestedOutput("output0")

start = time.time()
resp = client.infer("yolov8n", [inp], outputs=[out], headers=grpc_headers)
t_grpc = (time.time() - start) * 1000
output_grpc = resp.as_numpy("output0")
client.close()
print(f"✓ gRPC:          {t_grpc:8.2f} ms | Output: {output_grpc.shape}")

In [ ]:
# =============================================================================
# 3. REST (Binary) Protocol
# =============================================================================
tensor_binary = base_tensor[np.newaxis, ...].astype(np.float32).copy()
client = httpclient.InferenceServerClient(url=url)
inp = httpclient.InferInput("images", list(tensor_binary.shape), "FP32")
inp.set_data_from_numpy(tensor_binary, binary_data=True)
out = httpclient.InferRequestedOutput("output0", binary_data=True)

start = time.time()
resp = client.infer("yolov8n", [inp], outputs=[out], headers=http_headers)
t_binary = (time.time() - start) * 1000
output_binary = resp.as_numpy("output0")
print(f"✓ REST (Binary): {t_binary:8.2f} ms | Output: {output_binary.shape}")

In [ ]:
# =============================================================================
# Verify outputs match
# =============================================================================
print(f"Outputs match (JSON == gRPC):   {np.allclose(output_json, output_grpc)}")
print(f"Outputs match (gRPC == Binary): {np.allclose(output_grpc, output_binary)}")

In [ ]:
# =============================================================================
# Decode detections
# =============================================================================
COCO_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat",
    "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
    "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack",
    "umbrella", "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball",
    "kite", "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket",
    "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair",
    "couch", "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse",
    "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator",
    "book", "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush"
]

def decode_yolo_output(output, conf_threshold=0.25):
    pred = output[0].T
    boxes = pred[:, :4]
    class_scores = pred[:, 4:]
    class_ids = np.argmax(class_scores, axis=1)
    confidences = np.max(class_scores, axis=1)
    mask = confidences >= conf_threshold
    
    detections = []
    for i in np.where(mask)[0]:
        detections.append({
            "class": COCO_CLASSES[class_ids[i]],
            "confidence": round(float(confidences[i]), 3),
            "bbox_cxcywh": [round(float(x), 1) for x in boxes[i]]
        })
    return detections

detections = decode_yolo_output(output_grpc, conf_threshold=0.25)
print(f"Found {len(detections)} detections:")
for det in detections[:10]:
    print(f"  {det['class']:15} conf={det['confidence']:.2f}")